In [1]:
library(fixest)
library(modelsummary)
library(gt)
library(marginaleffects)
library(ggeffects)
library(ggplot2)
library(dplyr)
library(sjPlot)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Learn more about sjPlot with 'browseVignettes("sjPlot")'.



In [2]:
df_path='../../data/dropbox/LaborDivision/data/deriveddata/intl_noalpha/regression_table.csv'
data=read.csv(df_path)

In [3]:
head(data)

,ost_bk,cluster_id,cntry,role,year,dis,gender,yfp,npub,ST,funded
,<int>,<dbl>,<chr>,<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,29768070,65683243,JPN,F,2008,51,F,2008,5,Advanced,invalid
2,31240446,65683243,JPN,F,2008,51,F,2008,5,Advanced,invalid
3,29768070,79129017,JPN,M,2008,51,M,2000,98,Advanced,invalid
4,31240446,79129017,JPN,M,2008,51,M,2000,98,Advanced,invalid
5,39595623,79129017,JPN,F,2013,18,M,2000,98,Advanced,1
6,40710917,79129017,JPN,F,2013,51,M,2000,98,Advanced,1


In [4]:
data$npub=log(data$npub)
data$funded <- factor(data$funded, levels = c(0, 1), labels = c("NotFund", "Funded"))
data$fdep=ifelse(data$role == 'F', 1, 0)
data$ldep=ifelse(data$role == 'L', 1, 0)
data$mdep=ifelse(data$role == 'M', 1, 0)

In [5]:
head(data)

,ost_bk,cluster_id,cntry,role,year,dis,gender,yfp,npub,ST,funded,fdep,ldep,mdep
,<int>,<dbl>,<chr>,<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
1,29768070,65683243,JPN,F,2008,51,F,2008,1.609438,Advanced,NA,1,0,0
2,31240446,65683243,JPN,F,2008,51,F,2008,1.609438,Advanced,NA,1,0,0
3,29768070,79129017,JPN,M,2008,51,M,2000,4.584967,Advanced,NA,0,0,1
4,31240446,79129017,JPN,M,2008,51,M,2000,4.584967,Advanced,NA,0,0,1
5,39595623,79129017,JPN,F,2013,18,M,2000,4.584967,Advanced,Funded,1,0,0
6,40710917,79129017,JPN,F,2013,51,M,2000,4.584967,Advanced,Funded,1,0,0


In [6]:
unique_ids=unique(data$ost_bk)
sample_id=sample(unique_ids,1000000)
sample_data=data[data$ost_bk %in% sample_id,]

In [7]:
length(unique(data$ost_bk))

[1] 3195989

In [8]:
fund_data = data[!is.na(data$funded),]
unique_ids=unique(fund_data$ost_bk)
sample_id=sample(unique_ids,10000)
sample_fund=fund_data[fund_data$ost_bk %in% sample_id,]

In [9]:
head(sample_data)

,ost_bk,cluster_id,cntry,role,year,dis,gender,yfp,npub,ST,funded,fdep,ldep,mdep
,<int>,<dbl>,<chr>,<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
1,29768070,65683243,JPN,F,2008,51,F,2008,1.609438,Advanced,NA,1,0,0
2,31240446,65683243,JPN,F,2008,51,F,2008,1.609438,Advanced,NA,1,0,0
3,29768070,79129017,JPN,M,2008,51,M,2000,4.584967,Advanced,NA,0,0,1
4,31240446,79129017,JPN,M,2008,51,M,2000,4.584967,Advanced,NA,0,0,1
6,40710917,79129017,JPN,F,2013,51,M,2000,4.584967,Advanced,Funded,1,0,0
11,29768070,9735544,JPN,M,2008,51,M,1975,4.990433,Advanced,NA,0,0,1


In [10]:
head(sample_fund)

,ost_bk,cluster_id,cntry,role,year,dis,gender,yfp,npub,ST,funded,fdep,ldep,mdep
,<int>,<dbl>,<chr>,<chr>,<int>,<int>,<chr>,<dbl>,<dbl>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
106,77227626,87639223,JPN,L,2011,82,U,1986,5.780744,Advanced,NotFund,0,1,0
236,46698194,34324808,JPN,F,2016,93,F,1989,5.308268,Advanced,Funded,1,0,0
784,33588975,29996690,JPN,M,2010,31,M,1980,5.332719,Advanced,NotFund,0,0,1
862,73875605,74652763,JPN,L,2020,18,U,1985,6.133398,Advanced,Funded,0,1,0
1276,30394806,48447827,JPN,M,2008,12,M,1982,7.043160,Advanced,NotFund,0,0,1
1394,61221579,48447827,JPN,M,2012,23,M,1982,7.043160,Advanced,NotFund,0,0,1


## model with fixed effect with fixest package

In [11]:
model1 <- feglm(fdep~gender+npub+ST|ost_bk,data = sample_data,family = binomial())

In [12]:
summary(model1)

GLM estimation, family = binomial, Dep. Var.: fdep
Observations: 6,548,595 
Fixed-effects: ost_bk: 1,000,000
Standard-errors: Clustered (ost_bk) 
              Estimate Std. Error   t value  Pr(>|t|)    
genderM      -0.268232   0.003434  -78.1147 < 2.2e-16 ***
genderU      -0.296566   0.004873  -60.8550 < 2.2e-16 ***
npub         -0.226406   0.000813 -278.6062 < 2.2e-16 ***
STDeveloping  0.532390   0.008515   62.5204 < 2.2e-16 ***
STLagging     0.232364   0.007911   29.3736 < 2.2e-16 ***
STProficient  0.617728   0.004256  145.1512 < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -2,556,044.2   Adj. Pseudo R2: -0.270614
           BIC: 20,806,943.7     Squared Cor.:  0.081532

In [13]:
model2 <- feglm(fdep~gender+npub+ST+funded|ost_bk,data = sample_fund,family = binomial())
summary(model2)

GLM estimation, family = binomial, Dep. Var.: fdep
Observations: 66,696 
Fixed-effects: ost_bk: 10,000
Standard-errors: Clustered (ost_bk) 
              Estimate Std. Error   t value   Pr(>|t|)    
genderM      -0.274240   0.034762  -7.88898 3.0466e-15 ***
genderU      -0.320853   0.048908  -6.56033 5.3688e-11 ***
npub         -0.231189   0.007948 -29.08814  < 2.2e-16 ***
STDeveloping  0.680160   0.107630   6.31941 2.6256e-10 ***
STLagging     0.309797   0.092705   3.34176 8.3250e-04 ***
STProficient  0.668971   0.046204  14.47878  < 2.2e-16 ***
fundedFunded  1.021841   0.039549  25.83730  < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -25,316.2   Adj. Pseudo R2: -0.25321 
           BIC: 161,789.1     Squared Cor.:  0.093238

In [14]:
model3 <- feglm(fdep~gender+npub+ST+funded+ST*funded|ost_bk,data = sample_fund,family = binomial())
summary(model3)

GLM estimation, family = binomial, Dep. Var.: fdep
Observations: 66,696 
Fixed-effects: ost_bk: 10,000
Standard-errors: Clustered (ost_bk) 
                           Estimate Std. Error    t value   Pr(>|t|)    
genderM                   -0.274085   0.034814  -7.872870 3.4660e-15 ***
genderU                   -0.319896   0.048956  -6.534347 6.3888e-11 ***
npub                      -0.231645   0.007972 -29.056608  < 2.2e-16 ***
STDeveloping               0.532336   0.143307   3.714660 2.0348e-04 ***
STLagging                  0.197646   0.129670   1.524227 1.2745e-01    
STProficient               0.228154   0.092704   2.461109 1.3851e-02 *  
fundedFunded               0.851668   0.052007  16.376086  < 2.2e-16 ***
STDeveloping:fundedFunded  0.258525   0.208592   1.239384 2.1520e-01    
STLagging:fundedFunded     0.141277   0.179471   0.787188 4.3117e-01    
STProficient:fundedFunded  0.644394   0.114591   5.623419 1.8721e-08 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 

In [21]:
model4 <- feglm(mdep~gender+npub+ST|ost_bk,data = sample_data,family = binomial())
model5 <- feglm(mdep~gender+npub+ST+funded|ost_bk,data = sample_fund,family = binomial())
model6 <- feglm(mdep~gender+npub+ST+funded+ST*funded|ost_bk,data = sample_fund,family = binomial())

NOTE: 41,055 fixed-effects (82,110 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 301 fixed-effects (602 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 301 fixed-effects (602 observations) removed because of only 0 (or only 1) outcomes.



In [22]:
model7 <- feglm(ldep~gender+npub+ST|ost_bk,data = sample_data,family = binomial())
summary(model7)

GLM estimation, family = binomial, Dep. Var.: ldep
Observations: 6,548,595 
Fixed-effects: ost_bk: 1,000,000
Standard-errors: Clustered (ost_bk) 
              Estimate Std. Error  t value  Pr(>|t|)    
genderM       0.228005   0.003925  58.0894 < 2.2e-16 ***
genderU       0.194222   0.005299  36.6534 < 2.2e-16 ***
npub          0.495269   0.001147 431.7244 < 2.2e-16 ***
STDeveloping -0.595285   0.010548 -56.4365 < 2.2e-16 ***
STLagging    -0.420788   0.009619 -43.7434 < 2.2e-16 ***
STProficient -0.251302   0.004983 -50.4295 < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -2,421,330.7   Adj. Pseudo R2: -0.22248 
           BIC: 20,537,516.6     Squared Cor.:  0.121705

In [23]:
model8 <- feglm(ldep~gender+npub+ST+funded|ost_bk,data = sample_fund,family = binomial())
summary(model8)

GLM estimation, family = binomial, Dep. Var.: ldep
Observations: 66,696 
Fixed-effects: ost_bk: 10,000
Standard-errors: Clustered (ost_bk) 
              Estimate Std. Error  t value   Pr(>|t|)    
genderM       0.240532   0.039447  6.09767 1.0762e-09 ***
genderU       0.192491   0.052859  3.64158 2.7097e-04 ***
npub          0.503311   0.011607 43.36403  < 2.2e-16 ***
STDeveloping -0.546233   0.120272 -4.54165 5.5815e-06 ***
STLagging    -0.658970   0.110527 -5.96207 2.4906e-09 ***
STProficient -0.334659   0.050089 -6.68127 2.3689e-11 ***
fundedFunded  0.296767   0.039730  7.46958 8.0450e-14 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -24,317.1   Adj. Pseudo R2: -0.217761
           BIC: 159,790.9     Squared Cor.:  0.1253  

In [24]:
model9 <- feglm(ldep~gender+npub+ST+funded+ST*funded|ost_bk,data = sample_fund,family = binomial())
summary(model9)

GLM estimation, family = binomial, Dep. Var.: ldep
Observations: 66,696 
Fixed-effects: ost_bk: 10,000
Standard-errors: Clustered (ost_bk) 
                           Estimate Std. Error  t value   Pr(>|t|)    
genderM                    0.238910   0.039447  6.05650 1.3912e-09 ***
genderU                    0.190790   0.052885  3.60766 3.0897e-04 ***
npub                       0.504254   0.011628 43.36383  < 2.2e-16 ***
STDeveloping              -0.306809   0.168300 -1.82299 6.8305e-02 .  
STLagging                 -0.329277   0.141943 -2.31978 2.0353e-02 *  
STProficient              -0.247312   0.098246 -2.51728 1.1826e-02 *  
fundedFunded               0.390480   0.050049  7.80199 6.0941e-15 ***
STDeveloping:fundedFunded -0.483419   0.223397 -2.16395 3.0468e-02 *  
STLagging:fundedFunded    -0.643107   0.200872 -3.20157 1.3668e-03 ** 
STProficient:fundedFunded -0.144813   0.117639 -1.23100 2.1832e-01    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likeliho

In [34]:
models = list("(1)"=model1,"(2)"=model2,"(3)"=model3,"(1)"=model4,"(2)"=model5,"(3)"=model6,"(1)"=model7,"(2)"=model8,"(3)"=model9)
table = modelsummary(models, output='gt', stars=TRUE,
            coef_map=c('genderM'='Male','genderU'='Gender(Unknown)','npub'='Log no.pub',
                          'STProficient'='Proficient','STDeveloping'='Developing','STLagging'='Lagging',
                          'fundedFunded'='Funded',
                          'STProficient:fundedFunded'='Proficient × Funded',
                          'STDeveloping:fundedFunded'='Developing × Funded',
                          'STLagging:fundedFunded'='Lagging × Funded'),
            gof_map = c("nobs", "r.squared","adj.r.squared"))
table |> 
tab_spanner(label='First Author',columns=c(2:4)) |> 
tab_spanner(label='Middle Author',columns=c(5:7))|>
tab_spanner(label='Last Author',columns=c(8:10))|>
gtsave(file='regression.html')

In [17]:
summary(model3)

GLM estimation, family = binomial, Dep. Var.: fdep
Observations: 66,463 
Fixed-effects: ost_bk: 10,000
Standard-errors: Clustered (ost_bk) 
                           Estimate Std. Error    t value   Pr(>|t|)    
genderM                   -0.300455   0.034567  -8.691919  < 2.2e-16 ***
genderU                   -0.306774   0.048983  -6.262897 3.7789e-10 ***
npub                      -0.241064   0.008009 -30.098998  < 2.2e-16 ***
STDeveloping               0.471020   0.138894   3.391218 6.9583e-04 ***
STLagging                  0.289200   0.125951   2.296134 2.1668e-02 *  
STProficient               0.408935   0.091637   4.462558 8.0987e-06 ***
fundedFunded               0.822937   0.052201  15.764918  < 2.2e-16 ***
STDeveloping:fundedFunded  0.612312   0.219049   2.795320 5.1848e-03 ** 
STLagging:fundedFunded     0.021494   0.176714   0.121634 9.0319e-01    
STProficient:fundedFunded  0.477342   0.111358   4.286569 1.8145e-05 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 

In [15]:
options(repr.plot.width=12, repr.plot.height=4)

In [27]:
plot = plot_model(model3, type='pred', 
           terms=c("funded","ST [Advanced, Proficient,Developing,Lagging]","gender [F,M,U]"),
          title="",
          axis.title=c("",'Probability of being first author'),
          legend.title='Scientific Capacity',
          term_labels=c('F'='Female'))+
set_theme(axis.textsize=1.1)+
theme(axis.text.x = element_text(angle=0),
      panel.spacing=unit(2,"lines"),
      panel.background = element_blank(),
      panel.grid.major=element_line(linewidth = 0.1,color = "grey"),
      panel.grid.minor=element_line(linewidth = 0.1,color = "grey"),
      strip.background = element_blank()
      )
ggsave("MEF_First_plot.pdf", plot, width = 12, height = 4)

In [28]:
plot = plot_model(model6, type='pred', 
           terms=c("funded","ST [Advanced, Proficient,Developing,Lagging]","gender [F,M,U]"),
          title="",
          axis.title=c("",'Probability of being middle author'),
          legend.title='Scientific Capacity',
          term_labels=c('F'='Female'))+
set_theme(axis.textsize=1.1)+
theme(axis.text.x = element_text(angle=0),
      panel.spacing=unit(2,"lines"),
      panel.background = element_blank(),
      panel.grid.major=element_line(linewidth = 0.1,color = "grey"),
      panel.grid.minor=element_line(linewidth = 0.1,color = "grey"),
      strip.background = element_blank()
      )
ggsave("MEF_Middle_plot.pdf", plot, width = 12, height = 4)

In [29]:
plot = plot_model(model9, type='pred', 
           terms=c("funded","ST [Advanced, Proficient,Developing,Lagging]","gender [F,M,U]"),
          title="",
          axis.title=c("",'Probability of being last author'),
          legend.title='Scientific Capacity',
          term_labels=c('F'='Female'))+
set_theme(axis.textsize=1.1)+
theme(axis.text.x = element_text(angle=0),
      panel.spacing=unit(2,"lines"),
      panel.background = element_blank(),
      panel.grid.major=element_line(linewidth = 0.1,color = "grey"),
      panel.grid.minor=element_line(linewidth = 0.1,color = "grey"),
      strip.background = element_blank()
      )
ggsave("MEF_Last_plot.pdf", plot, width = 12, height = 4)